In [ ]:
import os
import time
import rospy
import cv2

import numpy as np
from geometry_msgs.msg import Twist

import bready.camera_utils.PyCamera as PyCamera
from bready.object_detection_tools import ObjectDetector

import ipywidgets.widgets as widgets
import IPython.display

In [ ]:
camera = PyCamera.URLCamera(
    camera_url = "tcp://127.0.0.1:5000",
    live_thread = True
)

In [ ]:
obj_model = ObjectDetector.Tester(
    'optimization',
    'detection_classes.txt',
    ["image_tensor"]
)

In [ ]:
def norm(vec):
    return np.sqrt(vec[0]**2 + vec[1]**2)

In [ ]:
def closest_detection(detections):
    closest_detection = None
    
    for det in detections:
        center = (det["detection_center_point_x"], det["detection_boxes"][2])
        
        if closest_detection is None:
            closest_detection = center
        elif norm(center) < norm(closest_detection):
            closest_detection = center
            
    return closest_detection

In [ ]:
def calc_ctrl_data(x, y):
    angular_data = (0.5-x)
    linear_data = (0.7 - min(y, 0.7)) * (4/0.7)
    
    return linear_data, angular_data

In [ ]:
rospy.init_node('Person_Detection', anonymous=True)

cmd_pub = rospy.Publisher('/cmd_vel', Twist, queue_size=1)

twist_data = Twist()
cmd_pub.publish(twist_data)

In [ ]:
image_widget1 = widgets.Image(format='jpeg')
image_widget2 = widgets.Image(format='jpeg')

IPython.display.display(widgets.HBox([image_widget1, image_widget2]))

In [ ]:
detect_label='person'

while(not rospy.is_shutdown()):
    try:
        image = camera.get_frame()
        
        if(image is None):
            break
            
        obj_model.execute(image)
        
        result_image = obj_model.getResultImage()
        
        xy_result = closest_detection(obj_model.getDetectedList2(detect_label))
        
        if(xy_result is not None):
            (x, y) = xy_result
        else:
            (x, y) = 0.5, 0.7
            
        cmd_vec_data = calc_ctrl_data(x, y)
        
        twist_data.linear.x = cmd_vec_data[0]
        twist_data.angular.z = cmd_vec_data[1]
        
        x = int(x * camera.camera_width)
        y = int(y * camera.camera_height)
        
        radius = int(max(camera.camera_width, camera.camera_height) / 28)
        thickness = int(max(camera.camera_width, camera.camera_height) / 74)
        
        result_image2 = cv2.circle(image.copy(), (x, y), radius, (0, 255, 0), thickness)
        
        tmpStream = cv2.imencode(".jpeg", result_image)[1].tostring()
        image_widget1.value = tmpStream
        
        tmpStream = cv2.imencode(".jpeg", result_image2)[1].tostring()
        image_widget2.value = tmpStream
        
        time.sleep(0.01)
        
        cmd_pub.publish(twist_data)
        
    except KeyboardInterrupt:
        print("종료")
        break